In [1]:
import os, sys, random, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

# Environment
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
print(f"Device  : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

PyTorch : 2.10.0+cu128
CUDA    : True
Device  : cuda
GPU     : Tesla T4


In [2]:
# Auto-discover the color/ data root
candidates = []
for root, dirs, files in os.walk("/kaggle/input"):
    for d in dirs:
        if d == "Corn_(maize)___healthy":
            candidates.append(Path(root) / d)

color_roots = [c.parent for c in candidates if c.parent.name == "color"]
assert color_roots, "No color/ folder found"
DATA_ROOT = color_roots[0]
print(f"DATA_ROOT = {DATA_ROOT}\n")

CLASS_DIRS = {
    "healthy":         "Corn_(maize)___healthy",
    "common_rust":     "Corn_(maize)___Common_rust_",
    "cercospora":      "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "northern_blight": "Corn_(maize)___Northern_Leaf_Blight",
}
IMG_EXTS = {".jpg", ".jpeg", ".png"}

# Same shuffle seed as NB1 -> identical splits
random.seed(42)
healthy_paths = sorted(
    str(p) for p in (DATA_ROOT / CLASS_DIRS["healthy"]).iterdir()
    if p.suffix.lower() in IMG_EXTS
)
random.shuffle(healthy_paths)

n = len(healthy_paths)
n_train, n_val = int(n * 0.80), int(n * 0.10)
healthy_train = healthy_paths[:n_train]
healthy_val   = healthy_paths[n_train : n_train + n_val]
healthy_test  = healthy_paths[n_train + n_val :]

diseased = []
for cls, folder in CLASS_DIRS.items():
    if cls == "healthy":
        continue
    paths = sorted(
        str(p) for p in (DATA_ROOT / folder).iterdir()
        if p.suffix.lower() in IMG_EXTS
    )
    diseased.extend((p, cls) for p in paths)

rows = []
for p in healthy_train: rows.append({"filepath": p, "label": 0, "split": "train", "class": "healthy"})
for p in healthy_val:   rows.append({"filepath": p, "label": 0, "split": "val",   "class": "healthy"})
for p in healthy_test:  rows.append({"filepath": p, "label": 0, "split": "test",  "class": "healthy"})
for p, cls in diseased: rows.append({"filepath": p, "label": 1, "split": "test",  "class": cls})

df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/splits.csv", index=False)

# Verify counts match NB1 exactly
expected = {"train": 929, "val": 116, "test": 2807}
for split, n_expected in expected.items():
    n_actual = (df["split"] == split).sum()
    assert n_actual == n_expected, f"{split}: expected {n_expected}, got {n_actual}"

print("Splits match NB1:")
print(df.groupby(["split", "label"]).size().unstack(fill_value=0))

DATA_ROOT = /kaggle/input/datasets/abdallahalidev/plantvillage-dataset/plantvillage dataset/color

Splits match NB1:
label    0     1
split           
test   117  2690
train  929     0
val    116     0


In [3]:
IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_WORKERS = 2

# CRITICAL: ImageNet normalization. Pretrained WideResNet50 expects this.
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

class MaizeDataset(torch.utils.data.Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["filepath"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return {"image": img, "label": int(row["label"]),
                "class": row["class"], "filepath": row["filepath"]}

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

train_ds = MaizeDataset(train_df, transform=transform)
val_ds   = MaizeDataset(val_df,   transform=transform)
test_ds  = MaizeDataset(test_df,  transform=transform)

# shuffle=False on train too — for PatchCore we want deterministic memory-bank order
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False,
                                           num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = torch.utils.data.DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                                           num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                                           num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

batch = next(iter(train_loader))
print(f"\nBatch shape : {tuple(batch['image'].shape)}     # expect (32, 3, 224, 224)")
print(f"Batch dtype : {batch['image'].dtype}")
print(f"Batch range : [{batch['image'].min():.3f}, {batch['image'].max():.3f}]   # expect roughly [-2.1, 2.6] after ImageNet norm")

Train: 929 | Val: 116 | Test: 2807

Batch shape : (32, 3, 224, 224)     # expect (32, 3, 224, 224)
Batch dtype : torch.float32
Batch range : [-2.067, 2.640]   # expect roughly [-2.1, 2.6] after ImageNet norm


In [4]:
# Container for captured intermediate features
features = {}

def get_hook(name):
    def hook(module, inputs, output):
        features[name] = output
    return hook

# Load pretrained WideResNet50
print("Loading WideResNet50 (ImageNet pretrained)...")
backbone = models.wide_resnet50_2(
    weights=models.Wide_ResNet50_2_Weights.IMAGENET1K_V1
)
backbone.eval()
backbone.to(DEVICE)

# Freeze: we never compute gradients on the backbone
for p in backbone.parameters():
    p.requires_grad = False

# Register hooks on layer2 and layer3
hook_handles = [
    backbone.layer2.register_forward_hook(get_hook("layer2")),
    backbone.layer3.register_forward_hook(get_hook("layer3")),
]

n_params = sum(p.numel() for p in backbone.parameters())
n_trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
print(f"Backbone params  : {n_params:,}")
print(f"Trainable params : {n_trainable}   # frozen — no training")

# Run ONE batch through to verify hooks fire and shapes are right
batch = next(iter(train_loader))
x = batch["image"].to(DEVICE)

with torch.no_grad():
    _ = backbone(x)

print(f"\nInput shape     : {tuple(x.shape)}            # (32, 3, 224, 224)")
print(f"layer2 features : {tuple(features['layer2'].shape)}    # expect (32, 512, 28, 28)")
print(f"layer3 features : {tuple(features['layer3'].shape)}   # expect (32, 1024, 14, 14)")
print(f"\nFeature dtypes  : layer2={features['layer2'].dtype}, layer3={features['layer3'].dtype}")
print(f"Feature device  : {features['layer2'].device}")

Loading WideResNet50 (ImageNet pretrained)...
Downloading: "https://download.pytorch.org/models/wide_resnet50_2-95faca4d.pth" to /root/.cache/torch/hub/checkpoints/wide_resnet50_2-95faca4d.pth


100%|██████████| 132M/132M [00:00<00:00, 172MB/s]


Backbone params  : 68,883,240
Trainable params : 0   # frozen — no training

Input shape     : (32, 3, 224, 224)            # (32, 3, 224, 224)
layer2 features : (32, 512, 28, 28)    # expect (32, 512, 28, 28)
layer3 features : (32, 1024, 14, 14)   # expect (32, 1024, 14, 14)

Feature dtypes  : layer2=torch.float32, layer3=torch.float32
Feature device  : cuda:0


In [5]:
def extract_patches(images, backbone, features_dict):
    """
    Extract PatchCore patch features for a batch of images.
    
    Pipeline:
      layer2 (B,512,28,28) + layer3 (B,1024,14,14)
      -> upsample layer3 to 28x28
      -> concat -> (B, 1536, 28, 28)
      -> 3x3 avg pool (neighborhood-aware patches) -> (B, 1536, 28, 28)
      -> reshape -> (B*784, 1536)
    """
    with torch.no_grad():
        _ = backbone(images)

    f2 = features_dict["layer2"]                          # (B, 512, 28, 28)
    f3 = features_dict["layer3"]                          # (B, 1024, 14, 14)

    f3_up = F.interpolate(f3, size=f2.shape[-2:],
                          mode="bilinear", align_corners=False)   # (B, 1024, 28, 28)

    concat = torch.cat([f2, f3_up], dim=1)                # (B, 1536, 28, 28)

    pooled = F.avg_pool2d(concat, kernel_size=3, stride=1, padding=1)   # (B, 1536, 28, 28)

    B, C, H, W = pooled.shape
    patches = pooled.permute(0, 2, 3, 1).reshape(B * H * W, C)        # (B*784, 1536)
    return patches


# Verify on one training batch
batch = next(iter(train_loader))
x = batch["image"].to(DEVICE, non_blocking=True)
patches = extract_patches(x, backbone, features)

print(f"Input batch     : {tuple(x.shape)}")
print(f"Patches output  : {tuple(patches.shape)}      # expect (32*784, 1536) = (25088, 1536)")
print(f"Patch dim       : {patches.shape[1]}                  # 512 (layer2) + 1024 (layer3) = 1536")
print(f"Patches per img : {patches.shape[0] // x.shape[0]}    # 28 * 28 = 784")
print(f"Patch dtype     : {patches.dtype}")
print(f"Patch device    : {patches.device}")

Input batch     : (32, 3, 224, 224)
Patches output  : (25088, 1536)      # expect (32*784, 1536) = (25088, 1536)
Patch dim       : 1536                  # 512 (layer2) + 1024 (layer3) = 1536
Patches per img : 784    # 28 * 28 = 784
Patch dtype     : torch.float32
Patch device    : cuda:0


In [6]:
print(f"Extracting patches from {len(train_ds)} healthy training images...")
print(f"Expected total patches: {len(train_ds)} * 784 = {len(train_ds) * 784:,}\n")

all_patches = []
t0 = time.time()

for batch in tqdm(train_loader, desc="Building memory bank"):
    x = batch["image"].to(DEVICE, non_blocking=True)
    patches = extract_patches(x, backbone, features)        # on GPU, (B*784, 1536)
    all_patches.append(patches.cpu())                       # move to CPU RAM
    
    # Free GPU memory between batches
    del patches
    
torch.cuda.empty_cache()

memory_bank = torch.cat(all_patches, dim=0)                 # (728224, 1536) on CPU
del all_patches

elapsed = time.time() - t0
mem_gb = memory_bank.element_size() * memory_bank.numel() / 1e9

print(f"\nMemory bank shape : {tuple(memory_bank.shape)}")
print(f"Memory bank size  : {mem_gb:.2f} GB")
print(f"Build time        : {elapsed:.1f}s")
print(f"Throughput        : {len(train_ds) / elapsed:.1f} images/sec")

# Quick statistics
print(f"\nFeature value range : [{memory_bank.min():.3f}, {memory_bank.max():.3f}]")
print(f"Feature mean (overall): {memory_bank.mean():.4f}")
print(f"Feature std (overall) : {memory_bank.std():.4f}")

Extracting patches from 929 healthy training images...
Expected total patches: 929 * 784 = 728,336



Building memory bank:   0%|          | 0/30 [00:00<?, ?it/s]


Memory bank shape : (728336, 1536)
Memory bank size  : 4.47 GB
Build time        : 12.6s
Throughput        : 73.9 images/sec

Feature value range : [0.000, 1.869]
Feature mean (overall): 0.0637
Feature std (overall) : 0.0927


In [7]:
from sklearn.random_projection import SparseRandomProjection

CORESET_RATIO  = 0.10
PROJECTION_DIM = 128

n_total   = memory_bank.shape[0]
n_coreset = int(n_total * CORESET_RATIO)
print(f"Coreset target: {CORESET_RATIO*100:.0f}% = {n_coreset:,} of {n_total:,} patches\n")

# 1. JL random projection: 1536 -> 128 dim
print(f"Sparse random projection: 1536 -> {PROJECTION_DIM} dim")
t0 = time.time()
projector = SparseRandomProjection(n_components=PROJECTION_DIM, random_state=42)
projected_np = projector.fit_transform(memory_bank.numpy())
projected = torch.from_numpy(projected_np).float().to(DEVICE)
del projected_np
print(f"Projection done: {time.time()-t0:.1f}s\n")

# 2. Greedy farthest-point sampling, kept entirely on GPU (no per-iter host sync)
print(f"Greedy farthest-point sampling on {DEVICE}...")
t0 = time.time()
n = projected.shape[0]

selected = torch.zeros(n_coreset, dtype=torch.long, device=DEVICE)

torch.manual_seed(42)
start_idx = torch.randint(0, n, (1,), device=DEVICE).item()
selected[0] = start_idx

# Initialize min_dist with squared distance to the first point
last = projected[start_idx].unsqueeze(0)
diff = projected - last
min_dist = (diff * diff).sum(dim=1)
min_dist[start_idx] = -float("inf")

for i in tqdm(range(1, n_coreset), desc="Coreset"):
    new_idx = torch.argmax(min_dist)        # 0-dim tensor, no .item() sync
    selected[i] = new_idx
    
    last = projected[new_idx].unsqueeze(0)
    diff = projected - last
    new_dist = (diff * diff).sum(dim=1)
    min_dist = torch.minimum(min_dist, new_dist)
    min_dist[new_idx] = -float("inf")

elapsed = time.time() - t0
print(f"Coreset selection: {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"Throughput: {n_coreset/elapsed:.0f} selections/sec")

# 3. Retrieve the ORIGINAL 1536-dim features at the selected indices
selected_cpu = selected.cpu()
coreset_bank = memory_bank[selected_cpu]    # (n_coreset, 1536), on CPU

print(f"\nCoreset bank shape : {tuple(coreset_bank.shape)}")
print(f"Coreset bank size  : {coreset_bank.element_size() * coreset_bank.numel() / 1e9:.3f} GB")
print(f"Compression ratio  : {n_total / n_coreset:.1f}x")

# Save the artifact for GitHub
torch.save({
    "coreset_bank":     coreset_bank,
    "selected_indices": selected_cpu,
    "n_total":          n_total,
    "coreset_ratio":    CORESET_RATIO,
    "projection_dim":   PROJECTION_DIM,
}, "/kaggle/working/patchcore_coreset.pt")
print(f"Saved -> /kaggle/working/patchcore_coreset.pt")

# Free intermediates
del projected
torch.cuda.empty_cache()

Coreset target: 10% = 72,833 of 728,336 patches

Sparse random projection: 1536 -> 128 dim
Projection done: 19.0s

Greedy farthest-point sampling on cuda...


Coreset:   0%|          | 0/72832 [00:00<?, ?it/s]

Coreset selection: 581.8s (9.7 min)
Throughput: 125 selections/sec

Coreset bank shape : (72833, 1536)
Coreset bank size  : 0.447 GB
Compression ratio  : 10.0x
Saved -> /kaggle/working/patchcore_coreset.pt


In [8]:
# Move coreset bank to GPU for nearest-neighbor lookup
coreset_bank_gpu = coreset_bank.to(DEVICE)
print(f"Coreset bank on GPU: {tuple(coreset_bank_gpu.shape)}")

# Free the full memory bank — we're done with it
del memory_bank
torch.cuda.empty_cache()

print(f"\nRunning PatchCore inference on {len(test_ds)} test images...")
t0 = time.time()
CHUNK_SIZE = 8   # images per GPU chunk inside each batch

all_scores, all_maps = [], []
all_labels, all_classes, all_paths = [], [], []

for batch in tqdm(test_loader, desc="PatchCore inference"):
    x = batch["image"].to(DEVICE, non_blocking=True)
    patches = extract_patches(x, backbone, features)   # (B*784, 1536) on GPU
    B = x.shape[0]
    
    # Compute min nearest-neighbor distance per patch in chunks
    chunk_min_dists = []
    for cs in range(0, B, CHUNK_SIZE):
        ce = min(cs + CHUNK_SIZE, B)
        chunk_patches = patches[cs * 784 : ce * 784]
        dists = torch.cdist(chunk_patches, coreset_bank_gpu)   # (chunk*784, 72833)
        min_d, _ = dists.min(dim=1)                            # (chunk*784,)
        chunk_min_dists.append(min_d)
        del dists
    
    all_min_dists = torch.cat(chunk_min_dists)                 # (B*784,)
    anomaly_maps  = all_min_dists.reshape(B, 28, 28)           # (B, 28, 28)
    image_scores  = anomaly_maps.amax(dim=(1, 2))              # (B,)
    
    all_scores.extend(image_scores.cpu().numpy())
    all_maps.append(anomaly_maps.cpu())
    all_labels.extend(batch["label"].tolist())
    all_classes.extend(batch["class"])
    all_paths.extend(batch["filepath"])
    
    del patches, all_min_dists, anomaly_maps, image_scores
    torch.cuda.empty_cache()

scores       = np.array(all_scores)
labels       = np.array(all_labels)
classes      = np.array(all_classes)
paths        = np.array(all_paths)
maps_tensor  = torch.cat(all_maps)                             # (2807, 28, 28)

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s ({elapsed/60:.1f} min)  |  {len(test_ds)/elapsed:.1f} img/s")
print(f"Anomaly maps : {tuple(maps_tensor.shape)}")
print(f"Score range  : [{scores.min():.4f}, {scores.max():.4f}]\n")

print(f"{'Class':<18} {'n':>6}  {'mean':>10}  {'median':>10}")
print("-" * 50)
for cls in ["healthy", "common_rust", "cercospora", "northern_blight"]:
    sub = scores[classes == cls]
    print(f"{cls:<18} {len(sub):>6}  {sub.mean():>10.4f}  {np.median(sub):>10.4f}")

# Save results for paper figures + GitHub
pd.DataFrame({
    "filepath": paths, "class": classes, "label": labels, "score": scores,
}).to_csv("/kaggle/working/patchcore_test_scores.csv", index=False)

torch.save({
    "scores":       torch.from_numpy(scores),
    "labels":       torch.from_numpy(labels),
    "classes":      classes.tolist(),
    "paths":        paths.tolist(),
    "anomaly_maps": maps_tensor,
}, "/kaggle/working/patchcore_results.pt")
print(f"\nSaved scores -> /kaggle/working/patchcore_test_scores.csv")
print(f"Saved maps   -> /kaggle/working/patchcore_results.pt")

Coreset bank on GPU: (72833, 1536)

Running PatchCore inference on 2807 test images...


PatchCore inference:   0%|          | 0/88 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78e7481918a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78e7481918a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
Exception ignored in:     <function _MultiProcessingDataLoaderIter.__del__ at 0x78e7481918a0>Exception ignored in: if w.is_alive():
<function _MultiProcessingDataLoaderIter.__del__ at 0x78e7481918a0>
Traceback (most recent call last):
 
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
     Traceback (most recent call last):
self._shutdown_workers


Done in 173.7s (2.9 min)  |  16.2 img/s
Anomaly maps : (2807, 28, 28)
Score range  : [2.6405, 5.2297]

Class                   n        mean      median
--------------------------------------------------
healthy               117      3.2296      3.1672
common_rust          1192      4.2090      4.1605
cercospora            513      3.9330      3.9091
northern_blight       985      3.9017      3.8731

Saved scores -> /kaggle/working/patchcore_test_scores.csv
Saved maps   -> /kaggle/working/patchcore_results.pt


In [9]:
from sklearn.metrics import roc_auc_score

# Overall AUROC
auroc_overall = roc_auc_score(labels, scores)
print(f"PatchCore Overall AUROC: {auroc_overall:.4f}")
print(f"  (test set: {(labels==0).sum()} healthy vs {(labels==1).sum()} diseased)\n")

# Per-class AUROC + comparison with NB1
healthy_scores = scores[classes == "healthy"]
nb1_per_class = {"common_rust": 0.9996, "cercospora": 0.9948, "northern_blight": 0.9874}
nb1_overall = 0.9942

print(f"{'Class':<18} {'n':>6}  {'PatchCore':>11}  {'NB1 (CAE)':>11}  {'Δ':>+8}")
print("-" * 65)
patchcore_per_class = {}
for cls in ["common_rust", "cercospora", "northern_blight"]:
    cls_scores = scores[classes == cls]
    y_score = np.concatenate([healthy_scores, cls_scores])
    y_true  = np.concatenate([np.zeros(len(healthy_scores)),
                              np.ones(len(cls_scores))])
    auc = roc_auc_score(y_true, y_score)
    patchcore_per_class[cls] = auc
    delta = auc - nb1_per_class[cls]
    print(f"{cls:<18} {len(cls_scores):>6}  {auc:>11.4f}  {nb1_per_class[cls]:>11.4f}  {delta:>+8.4f}")

print(f"\n{'Overall':<18} {len(scores):>6}  "
      f"{auroc_overall:>11.4f}  {nb1_overall:>11.4f}  {auroc_overall - nb1_overall:>+8.4f}")

PatchCore Overall AUROC: 0.9536
  (test set: 117 healthy vs 2690 diseased)



ValueError: Sign not allowed in string format specifier

In [ ]:
from sklearn.metrics import roc_auc_score

# Overall AUROC
auroc_overall = roc_auc_score(labels, scores)
print(f"PatchCore Overall AUROC: {auroc_overall:.4f}")
print(f"  (test set: {(labels==0).sum()} healthy vs {(labels==1).sum()} diseased)\n")

healthy_scores = scores[classes == "healthy"]
nb1_per_class  = {"common_rust": 0.9996, "cercospora": 0.9948, "northern_blight": 0.9874}
nb1_overall    = 0.9942

# Header (no leading + in format spec — that's what caused the error)
print(f"{'Class':<18} {'n':>6}  {'PatchCore':>11}  {'NB1 (CAE)':>11}  {'Delta':>9}")
print("-" * 65)

patchcore_per_class = {}
for cls in ["common_rust", "cercospora", "northern_blight"]:
    cls_scores = scores[classes == cls]
    y_score = np.concatenate([healthy_scores, cls_scores])
    y_true  = np.concatenate([np.zeros(len(healthy_scores)),
                              np.ones(len(cls_scores))])
    auc = roc_auc_score(y_true, y_score)
    patchcore_per_class[cls] = auc
    delta = auc - nb1_per_class[cls]
    # The +sign goes on the value itself (numeric format), not the header
    print(f"{cls:<18} {len(cls_scores):>6}  {auc:>11.4f}  {nb1_per_class[cls]:>11.4f}  {delta:>+9.4f}")

delta_overall = auroc_overall - nb1_overall
print(f"\n{'Overall':<18} {len(scores):>6}  "
      f"{auroc_overall:>11.4f}  {nb1_overall:>11.4f}  {delta_overall:>+9.4f}")

In [ ]:
import matplotlib.pyplot as plt

class_colors = {
    "healthy":         "#2ca02c",
    "common_rust":     "#d62728",
    "cercospora":      "#ff7f0e",
    "northern_blight": "#9467bd",
}
class_order = ["healthy", "common_rust", "cercospora", "northern_blight"]

fig, ax = plt.subplots(figsize=(11, 5.5))
bins = np.linspace(scores.min(), scores.max(), 60)

for cls in class_order:
    sub = scores[classes == cls]
    ax.hist(sub, bins=bins, density=True, alpha=0.55,
            label=f"{cls} (n={len(sub)}, mean={sub.mean():.3f})",
            color=class_colors[cls], edgecolor="black", linewidth=0.3)

thresh = np.percentile(scores[classes == "healthy"], 95)
ax.axvline(thresh, color="black", linestyle="--", alpha=0.6,
           label=f"95th pct of healthy = {thresh:.3f}")

ax.set_xlabel("PatchCore anomaly score (max patch distance)")
ax.set_ylabel("Density")
ax.set_title("PatchCore anomaly score by class — PlantVillage maize test set")
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/patchcore_score_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from scipy.ndimage import gaussian_filter

# Pick one representative image per class (same seed as NB1 figure for visual parity)
rng = np.random.RandomState(0)
example_idx = {cls: rng.choice(np.where(classes == cls)[0]) for cls in class_order}

maps_np = maps_tensor.numpy()   # (2807, 28, 28)

fig, axes = plt.subplots(4, 3, figsize=(11, 14))

for row, cls in enumerate(class_order):
    idx = example_idx[cls]
    path = paths[idx]
    score_val = scores[idx]
    amap = maps_np[idx]                            # (28, 28)
    
    # Original at 224x224
    img_pil = Image.open(path).convert("RGB").resize((224, 224))
    img_np = np.asarray(img_pil) / 255.0
    
    # Upsample anomaly map to 224x224 + Gaussian smooth for cleaner visualization
    amap_t = torch.from_numpy(amap).unsqueeze(0).unsqueeze(0)
    amap_up = F.interpolate(amap_t, size=(224, 224), mode="bilinear",
                            align_corners=False).squeeze().numpy()
    amap_up = gaussian_filter(amap_up, sigma=4)
    
    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title(f"{cls}\noriginal", fontsize=10)
    axes[row, 0].axis("off")
    
    im = axes[row, 1].imshow(amap_up, cmap="hot")
    axes[row, 1].set_title(f"PatchCore anomaly map\nimage score = {score_val:.3f}", fontsize=10)
    axes[row, 1].axis("off")
    plt.colorbar(im, ax=axes[row, 1], fraction=0.046, pad=0.04)
    
    axes[row, 2].imshow(img_np)
    axes[row, 2].imshow(amap_up, cmap="hot", alpha=0.5)
    axes[row, 2].set_title("overlay", fontsize=10)
    axes[row, 2].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/patchcore_anomaly_maps.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Use the anomaly maps already in memory
maps_np = maps_tensor.numpy()   # (2807, 28, 28)

def score_max(m):
    return m.reshape(m.shape[0], -1).max(axis=1)

def score_mean(m):
    return m.reshape(m.shape[0], -1).mean(axis=1)

def score_topk_mean(m, k_pct):
    flat = m.reshape(m.shape[0], -1)
    k = max(1, int(flat.shape[1] * k_pct))
    sorted_desc = np.sort(flat, axis=1)[:, ::-1]
    return sorted_desc[:, :k].mean(axis=1)

def score_center_max(m, margin):
    return m[:, margin:m.shape[1]-margin, margin:m.shape[2]-margin].max(axis=(1, 2))

def score_center_mean(m, margin):
    return m[:, margin:m.shape[1]-margin, margin:m.shape[2]-margin].mean(axis=(1, 2))

strategies = {
    "max (current)":       score_max(maps_np),
    "mean":                score_mean(maps_np),
    "top-10% mean":        score_topk_mean(maps_np, 0.10),
    "top-1% mean":         score_topk_mean(maps_np, 0.01),
    "center max (m=4)":    score_center_max(maps_np, 4),
    "center mean (m=4)":   score_center_mean(maps_np, 4),
    "center top-10% mean": None,   # combine: center then top-k
}

# combined: crop edges, then top-10% mean
center = maps_np[:, 4:-4, 4:-4]
strategies["center top-10% mean"] = score_topk_mean(center, 0.10)

print(f"{'Strategy':<22} {'Overall':>8}  {'rust':>7}  {'cerc':>7}  {'blight':>8}  {'Δ vs CAE':>10}")
print("-" * 70)

cae_overall = 0.9942
healthy_mask = (classes == "healthy")

for name, sc in strategies.items():
    overall = roc_auc_score(labels, sc)
    
    per_class = {}
    for cls in ["common_rust", "cercospora", "northern_blight"]:
        cls_mask = (classes == cls)
        y_score = np.concatenate([sc[healthy_mask], sc[cls_mask]])
        y_true  = np.concatenate([np.zeros(healthy_mask.sum()),
                                   np.ones(cls_mask.sum())])
        per_class[cls] = roc_auc_score(y_true, y_score)
    
    delta_cae = overall - cae_overall
    print(f"{name:<22} {overall:>8.4f}  "
          f"{per_class['common_rust']:>7.4f}  "
          f"{per_class['cercospora']:>7.4f}  "
          f"{per_class['northern_blight']:>8.4f}  "
          f"{delta_cae:>+10.4f}")

In [ ]:
# The new official PatchCore score is mean-aggregated patch distance
official_scores = score_mean(maps_np)

# Save updated CSV (overwrites)
pd.DataFrame({
    "filepath":  paths,
    "class":     classes,
    "label":     labels,
    "score_max":  score_max(maps_np),    # original (for ablation reference)
    "score_mean": official_scores,        # the new official score
}).to_csv("/kaggle/working/patchcore_test_scores.csv", index=False)

# Update the saved results bundle
torch.save({
    "scores_max":   torch.from_numpy(score_max(maps_np)),
    "scores_mean":  torch.from_numpy(official_scores),
    "labels":       torch.from_numpy(labels),
    "classes":      classes.tolist(),
    "paths":        paths.tolist(),
    "anomaly_maps": maps_tensor,
}, "/kaggle/working/patchcore_results.pt")

# Re-do the figures with the official mean-based score
fig, ax = plt.subplots(figsize=(11, 5.5))
bins = np.linspace(official_scores.min(), official_scores.max(), 60)
for cls in class_order:
    sub = official_scores[classes == cls]
    ax.hist(sub, bins=bins, density=True, alpha=0.55,
            label=f"{cls} (n={len(sub)}, mean={sub.mean():.3f})",
            color=class_colors[cls], edgecolor="black", linewidth=0.3)
thresh = np.percentile(official_scores[classes == "healthy"], 95)
ax.axvline(thresh, color="black", linestyle="--", alpha=0.6,
           label=f"95th pct of healthy = {thresh:.3f}")
ax.set_xlabel("PatchCore anomaly score (mean patch distance)")
ax.set_ylabel("Density")
ax.set_title("PatchCore-mean anomaly score by class — PlantVillage maize")
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/patchcore_score_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

# ROC curves with mean-based score
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
overall_auc = roc_auc_score(labels, official_scores)
fpr, tpr, _ = roc_curve(labels, official_scores)
axes[0].plot(fpr, tpr, label=f"all diseases (AUROC={overall_auc:.4f})", color="black", linewidth=2)
axes[1].plot(fpr, tpr, color="black", linewidth=2)

healthy_sc = official_scores[classes == "healthy"]
for cls in ["common_rust", "cercospora", "northern_blight"]:
    cls_sc = official_scores[classes == cls]
    y_score = np.concatenate([healthy_sc, cls_sc])
    y_true  = np.concatenate([np.zeros(len(healthy_sc)), np.ones(len(cls_sc))])
    fpr_c, tpr_c, _ = roc_curve(y_true, y_score)
    auc_c = roc_auc_score(y_true, y_score)
    axes[0].plot(fpr_c, tpr_c, label=f"{cls} (AUROC={auc_c:.4f})", color=class_colors[cls], alpha=0.85)
    axes[1].plot(fpr_c, tpr_c, color=class_colors[cls], alpha=0.85)

for ax in axes:
    ax.plot([0, 1], [0, 1], "k:", alpha=0.4)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.grid(alpha=0.3)
axes[0].set_title("ROC — full")
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1.01)
axes[0].legend(loc="lower right", fontsize=9)
axes[1].set_title("ROC — zoomed (FPR 0–0.1)")
axes[1].set_xlim(0, 0.1); axes[1].set_ylim(0.85, 1.005)
plt.tight_layout()
plt.savefig("/kaggle/working/patchcore_roc.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nOfficial PatchCore numbers (mean-aggregated patch distance):")
print(f"  Overall AUROC: {overall_auc:.4f}")
for cls in ["common_rust", "cercospora", "northern_blight"]:
    cls_sc = official_scores[classes == cls]
    y_score = np.concatenate([healthy_sc, cls_sc])
    y_true  = np.concatenate([np.zeros(len(healthy_sc)), np.ones(len(cls_sc))])
    auc_c = roc_auc_score(y_true, y_score)
    print(f"  {cls:<18} AUROC: {auc_c:.4f}")

In [ ]:
# Compute mean RGB for an image (one 3D vector per image)
def compute_mean_rgb_batch(filepaths, desc="Mean RGB"):
    means = []
    for fp in tqdm(filepaths, desc=desc):
        img = np.asarray(Image.open(fp).convert("RGB")).astype(np.float32)
        means.append(img.reshape(-1, 3).mean(axis=0))
    return np.array(means)

# Healthy training centroid in RGB space
train_paths_list = df[df["split"] == "train"]["filepath"].tolist()
print(f"Computing mean RGB for {len(train_paths_list)} healthy training images...")
train_means = compute_mean_rgb_batch(train_paths_list, desc="Train")
healthy_centroid = train_means.mean(axis=0)
print(f"\nHealthy training centroid: R={healthy_centroid[0]:.1f}  G={healthy_centroid[1]:.1f}  B={healthy_centroid[2]:.1f}")

# Test images
test_paths_list  = df[df["split"] == "test"]["filepath"].tolist()
test_labels      = df[df["split"] == "test"]["label"].values
test_classes     = df[df["split"] == "test"]["class"].values

print(f"\nComputing mean RGB for {len(test_paths_list)} test images...")
test_means = compute_mean_rgb_batch(test_paths_list, desc="Test")

# Anomaly score = L2 distance from healthy centroid in RGB-mean space
brightness_scores = np.linalg.norm(test_means - healthy_centroid, axis=1)

# Overall AUROC
brightness_overall = roc_auc_score(test_labels, brightness_scores)

# Per-class
healthy_b_scores = brightness_scores[test_classes == "healthy"]

print(f"\n=== Brightness-only baseline (mean-RGB distance from healthy centroid) ===\n")
print(f"{'Class':<18} {'n':>6}  {'Brightness':>12}  {'PatchCore':>11}  {'CAE':>8}")
print("-" * 65)

cae_per_class = {"common_rust": 0.9996, "cercospora": 0.9948, "northern_blight": 0.9874}
patchcore_per_class = {"common_rust": 1.0000, "cercospora": 0.9994, "northern_blight": 0.9990}

for cls in ["common_rust", "cercospora", "northern_blight"]:
    cls_scores = brightness_scores[test_classes == cls]
    y_score = np.concatenate([healthy_b_scores, cls_scores])
    y_true  = np.concatenate([np.zeros(len(healthy_b_scores)),
                               np.ones(len(cls_scores))])
    auc = roc_auc_score(y_true, y_score)
    print(f"{cls:<18} {len(cls_scores):>6}  {auc:>12.4f}  "
          f"{patchcore_per_class[cls]:>11.4f}  {cae_per_class[cls]:>8.4f}")

print(f"\n{'Overall':<18} {len(brightness_scores):>6}  "
      f"{brightness_overall:>12.4f}  {0.9995:>11.4f}  {0.9942:>8.4f}")

# Save baseline scores
pd.DataFrame({
    "filepath": test_paths_list,
    "class":    test_classes,
    "label":    test_labels,
    "score":    brightness_scores,
}).to_csv("/kaggle/working/brightness_baseline_scores.csv", index=False)
print(f"\nSaved -> /kaggle/working/brightness_baseline_scores.csv")